# Fine-tuning Phi-4 Mini as a Telecom Root Cause Agent (QLoRA on Colab free T4)

**Before running:** Runtime -> Change runtime type -> T4 GPU (free tier is enough for this).

**Upload `training_data.jsonl`** (produced by `generate_training_data.py` on your local
machine) using the file upload cell below.

This notebook: installs dependencies -> loads Phi-4 Mini in 4-bit -> fine-tunes with
LoRA (QLoRA) on your telecom evidence -> root-cause dataset -> merges the adapter into
the base model -> converts to GGUF (quantized) so it can run on YOUR 8GB CPU machine
afterward via `llama-cpp-python`.

If `microsoft/Phi-4-mini-instruct` isn't the exact model name when you run this
(model names on Hugging Face can change), search "Phi-4-mini" on huggingface.co
and swap `MODEL_NAME` below.


## 1. Install dependencies

In [ ]:
!pip install -q -U transformers accelerate peft trl bitsandbytes datasets huggingface_hub


In [ ]:
import torch
print("GPU available:", torch.cuda.is_available())
print("GPU name:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE -- go to Runtime > Change runtime type > T4 GPU")


## 2. Upload your training_data.jsonl

In [ ]:
from google.colab import files
uploaded = files.upload()  # select training_data.jsonl from your computer


## 3. Load and inspect the dataset

In [ ]:
from datasets import load_dataset

dataset = load_dataset("json", data_files="training_data.jsonl", split="train")
print(f"{len(dataset)} examples loaded")
print(dataset[0])

# 90/10 train/val split
split = dataset.train_test_split(test_size=0.1, seed=42)
train_dataset, eval_dataset = split["train"], split["test"]
print(f"Train: {len(train_dataset)}, Val: {len(eval_dataset)}")


## 4. Load Phi-4 Mini in 4-bit (QLoRA)

4-bit quantization is what makes fine-tuning a ~3.8B model fit comfortably in the
free T4's 15GB VRAM.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL_NAME = "microsoft/Phi-4-mini-instruct"  # verify this matches the current HF repo name

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)
print("Model loaded.")


## 5. Configure LoRA

In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    # Standard attention + MLP projection targets for Phi-family architectures.
    # If this errors with "target module not found", print(model) to see the
    # actual layer names and adjust this list to match.
    target_modules=["qkv_proj", "o_proj", "gate_up_proj", "down_proj"],
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()


## 6. Format the dataset using the model's chat template

In [ ]:
def formatting_func(example):
    text = tokenizer.apply_chat_template(example["messages"], tokenize=False, add_generation_prompt=False)
    return text

# Quick check on one example
print(formatting_func(train_dataset[0])[:500])


## 7. Train

In [ ]:
from trl import SFTTrainer, SFTConfig

training_args = SFTConfig(
    output_dir="./phi4-telecom-rootcause",
    num_train_epochs=3,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    logging_steps=5,
    eval_strategy="steps",
    eval_steps=20,
    save_strategy="epoch",
    bf16=True,
    max_seq_length=1024,
    packing=False,
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    formatting_func=formatting_func,
)

trainer.train()


## 8. Save the LoRA adapter

In [ ]:
ADAPTER_PATH = "./phi4-telecom-rootcause-adapter"
trainer.save_model(ADAPTER_PATH)
tokenizer.save_pretrained(ADAPTER_PATH)
print(f"Adapter saved to {ADAPTER_PATH}")


## 9. Merge the adapter into the base model

This produces a standalone model (no PEFT/adapter needed at inference time) --
required before GGUF conversion.

In [ ]:
from peft import PeftModel

# Reload base model in full precision for a clean merge (not 4-bit -- merging
# into a quantized base isn't reliable)
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, torch_dtype=torch.bfloat16, device_map="auto", trust_remote_code=True,
)
merged_model = PeftModel.from_pretrained(base_model, ADAPTER_PATH)
merged_model = merged_model.merge_and_unload()

MERGED_PATH = "./phi4-telecom-rootcause-merged"
merged_model.save_pretrained(MERGED_PATH)
tokenizer.save_pretrained(MERGED_PATH)
print(f"Merged model saved to {MERGED_PATH}")


## 10. Convert to GGUF (quantized) for CPU inference on your local 8GB machine

This clones llama.cpp and uses its conversion + quantization scripts. Q4_K_M is a
good balance of size vs. quality for an 8GB RAM machine.

In [ ]:
!git clone --depth 1 https://github.com/ggerganov/llama.cpp.git
!pip install -q -r llama.cpp/requirements.txt


In [ ]:
!python llama.cpp/convert_hf_to_gguf.py {MERGED_PATH} --outfile phi4-telecom-rootcause-f16.gguf --outtype f16


In [ ]:
# Build llama.cpp's quantize tool, then quantize down to Q4_K_M (much smaller,
# runs comfortably on 8GB RAM)
!cmake -B llama.cpp/build -S llama.cpp
!cmake --build llama.cpp/build --target llama-quantize -j
!./llama.cpp/build/bin/llama-quantize phi4-telecom-rootcause-f16.gguf phi4-telecom-rootcause-Q4_K_M.gguf Q4_K_M


## 11. Download the quantized model

In [ ]:
from google.colab import files
files.download("phi4-telecom-rootcause-Q4_K_M.gguf")


## Next steps (back on your local machine)

1. Move `phi4-telecom-rootcause-Q4_K_M.gguf` into `telecom_sim_v2/agents/models/`
2. `pip install llama-cpp-python --break-system-packages`
3. Run `python orchestrator_llm.py --tower T0000` (uses this file automatically --
   see the `Phi4LocalProvider` in `llm_providers.py`)

If quality isn't great with only ~50-500 training examples (depends on how many
failure events your simulator generated), that's expected -- this is a small,
synthetic dataset. Consider re-running `generate_training_data.py` after bumping
`FAILURES_PER_TOWER_PER_YEAR` in `config.py` for more training examples, or
increasing `num_train_epochs` above (watch eval loss for overfitting if you do).
